# Test Atomic Agent Marketplace Deployment

This notebook tests the marketplace agent deployment functionality with the atomic agent.

In [ ]:
import mlrun
import mlrun.marketplace
from mlrun.marketplace.agent import MarketplaceBackend

## Setup

1. Upload `atomic-agent.yaml` to your Jupyter workspace
2. Upload your `atomic-agent.tar.gz` source archive
3. Upload your `requirements.txt` file
4. Set the paths below

In [ ]:
# Configure paths (update these to match your setup)
YAML_DIR = "./"  # Directory containing atomic-agent.yaml
SOURCE_TAR = "./atomic-agent.tar.gz"  # Path to source archive
REQUIREMENTS_FILE = "./requirements.txt"  # Path to requirements file
PROJECT_NAME = "agent-demo"

# Required secrets (update with your actual values)
OPENAI_BASE_URL = "https://openai.prod.ai-gateway.quantumblack.com/a192ddc9-2002-4302-9840-26ad5eb678da/v1"
OPENAI_API_KEY = "YOUR_JWT_TOKEN_HERE"  # Replace with actual token

## Configure Backend to Load from Local YAML

In [ ]:
# Point backend to directory containing atomic-agent.yaml
MarketplaceBackend.set_yaml_directory(YAML_DIR)
print(f"Backend configured to load from: {YAML_DIR}")

## Upload Source Archive to Project

This uploads the tar.gz to MLRun's artifact store so it can be used during deployment.

In [ ]:
# Get or create project
project = mlrun.get_or_create_project(PROJECT_NAME, context="./")

# Upload source archive as artifact
atomic_agent_artifact = project.log_artifact(
    "atomic-agent-source",
    local_path=SOURCE_TAR
)

source_url = atomic_agent_artifact.target_path
print(f"Source uploaded to: {source_url}")

## Import Agent and View Info

In [ ]:
# Import agent from marketplace
agent = mlrun.import_agent("atomic-agent")

# Display agent information
agent.info()

## Deploy Agent

This will:
1. Build base image with requirements (slow, ~10 minutes)
2. Build image with source code
3. Deploy the application
4. Create API gateway

In [ ]:
# Deploy the agent
url = agent.deploy(
    project=PROJECT_NAME,
    source=source_url,
    requirements=REQUIREMENTS_FILE,
    gateway_config={
        "name": "atomic-agent-gw",
        "authentication_mode": "none",
    },
    # Required secrets
    OPENAI_BASE_URL=OPENAI_BASE_URL,
    OPENAI_API_KEY=OPENAI_API_KEY,
)

print(f"\n✅ Agent deployed successfully!")
print(f"URL: {url}")

## Test the Deployed Agent

In [ ]:
import requests
import urllib3
import uuid

# Disable SSL warnings for self-signed certs
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Construct full URL (add https:// if not present)
if not url.startswith("http"):
    test_url = f"https://{url}"
else:
    test_url = url

# A2A JSON-RPC request format
payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "params": {
        "message": {
            "messageId": str(uuid.uuid4()),
            "role": "user",
            "parts": [
                {
                    "text": "Hello! Can you help me with a simple task?"
                }
            ]
        }
    },
    "id": 1
}

print(f"Testing agent at: {test_url}\n")

response = requests.post(test_url, json=payload, verify=False)

print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    if "result" in result:
        print("\n✅ Agent Response:")
        artifacts = result["result"].get("artifacts", [])
        if artifacts:
            for artifact in artifacts:
                parts = artifact.get("parts", [])
                for part in parts:
                    if part.get("kind") == "text":
                        print(part.get("text"))
    else:
        print("\n❌ Error in response:")
        print(result)
else:
    print(f"\n❌ Request failed:")
    print(response.text)

## Test Redeployment (Should Be Fast!)

Redeploy with different config - should reuse cached images.

In [ ]:
# Redeploy with same agent instance - should be much faster!
url2 = agent.deploy(
    project=PROJECT_NAME,
    source=source_url,
    requirements=REQUIREMENTS_FILE,
    gateway_config={
        "name": "atomic-agent-gw",
        "authentication_mode": "none",
    },
    OPENAI_BASE_URL=OPENAI_BASE_URL,
    OPENAI_API_KEY=OPENAI_API_KEY,
)

print(f"\n✅ Agent redeployed (should have been faster!)")
print(f"URL: {url2}")

## Test Convenience Function

Test the one-call `deploy_agent()` function.

In [ ]:
# One-call deployment
url3 = mlrun.deploy_agent(
    "atomic-agent",
    project=PROJECT_NAME,
    source=source_url,
    requirements=REQUIREMENTS_FILE,
    gateway_config={
        "name": "atomic-agent-gw",
        "authentication_mode": "none",
    },
    OPENAI_BASE_URL=OPENAI_BASE_URL,
    OPENAI_API_KEY=OPENAI_API_KEY,
)

print(f"\n✅ Agent deployed via convenience function!")
print(f"URL: {url3}")